# Task 1.1: Core Contribution / Architecture

## Step-by-Step Method Description

**Paper:** Multiple Incremental Decremental Learning of Support Vector Machines (Karasuyama & Takeuchi, NeurIPS 2009)

---

### Step 1: SVM Formulation and KKT Index Sets
- **Description:** The method builds on the standard SVM formulation. Given training data {(x_i, y_i)}, the discriminant function is f(x) = sum_i alpha_i y_i K(x, x_i) + b. From the KKT conditions (Eq. 1a-1d), the paper defines three index sets: **O** (outliers, alpha_i=0), **M** (margin support vectors, 0 < alpha_i < C), and **I** (in-bound support vectors, alpha_i = C).
- **Reference:** Section 2, Equations (1a)-(1c), (2a)-(2c)
- **Purpose:** These index sets partition the training points and determine which parameters participate in the linear system that governs updates.

### Step 2: Define Add/Remove Sets and Initialization
- **Description:** When adding m points and removing l points, the paper defines A = {n+1, ..., n+m} (indices to add) and R (indices to remove). Elements of R are removed from M, I, O. New points in A are initialized with alpha_i = 0. Points in A with y_i f(x_i) > 1 move to O; those with y_i f(x_i) = 1 move to M.
- **Reference:** Section 3.2, first paragraph
- **Purpose:** Prepares the parameter space for simultaneous multi-point updates.

### Step 3: Choose Update Directions
- **Description:** For points to remove, the direction is Delta_alpha_R = -eta * alpha_R (shortest path to zero). For points to add, the direction is Delta_alpha_A = eta * (C*1 - alpha_A). This is the shortest path if all new points end at alpha_i = C at optimality.
- **Reference:** Equations (3) and (4)
- **Purpose:** Defines the direction of movement in the multi-dimensional coefficient space; this shorter path reduces the number of breakpoints compared to coordinate-wise updates.

### Step 4: Solve Linear System for Delta_b and Delta_alpha_M
- **Description:** To keep KKT conditions satisfied, the method solves the linear system (7). Substituting (3) and (4) into (7) yields [Delta_b; Delta_alpha_M] = eta * phi, where phi = -M^{-1} * [y_A^T y_R^T; Q_M,A Q_M,R] * [C*1 - alpha_A; -alpha_R].
- **Reference:** Equations (7) and (10)
- **Purpose:** Computes how the bias and margin coefficients must change so that equality constraints remain satisfied along the path.

### Step 5: Compute Step Length eta
- **Description:** The change in y_i f(x_i) is linear in eta (Eq. 11). The algorithm computes the largest eta >= 0 at which any inequality (8a)-(8c) or (9) becomes equality. The step length is eta = min({eta_tilde in H | eta_tilde >= 0} union {1}).
- **Reference:** Equations (8), (9), (11)
- **Purpose:** Advances along the path until a constraint is hit (a breakpoint), without violating KKT conditions.

### Step 6: Update Parameters and Index Sets
- **Description:** Using the chosen eta, update alpha_M, b, alpha_A, and alpha_R. If any point hits a constraint boundary, update M, O, I, and A accordingly. Recompute phi and psi for the next segment.
- **Reference:** Section 3.2, paragraph after Eq. (11)
- **Purpose:** Moves the solution along the path; each constraint hit is a breakpoint where the linear system size may change.

### Step 7: Handle Empty Margin (if M becomes empty)
- **Description:** When M is empty, the bias is only constrained to an interval (Eq. 12). The paper describes how to find a new margin point or trace piece-wise linear bounds until the bias is uniquely determined again.
- **Reference:** Section 3.3, Equations (12)-(13), Figure 1
- **Purpose:** Ensures the algorithm continues correctly when no points lie exactly on the margin.

### Step 8: Repeat Until Termination
- **Description:** Repeat steps 4-7 until eta = 1, meaning all new points satisfy optimality and alpha_R = 0. The algorithm then terminates.
- **Reference:** Section 3.2
- **Purpose:** Produces the updated SVM solution after adding and removing the specified points.

---

## Final Summary

**Problem solved:** Efficiently updating an SVM when multiple data points are added and/or removed simultaneously, as required in online learning.

**Claimed advantage:** By following a shorter path in the coefficient space (multi-parametric programming) instead of coordinate-wise updates, the algorithm reduces the number of breakpoints and thus computational cost compared to repeatedly applying the single-point incremental-decremental algorithm.